In [ ]:
!pip install razdel

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 87.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
from google.colab import drive
import logging
import functools
import re
import time
from typing import List, Dict, Tuple, Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string
from razdel import tokenize
import spacy
import openai
from google.colab import userdata
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import json
from datetime import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets
from typing import List
from typing import Any
from dataclasses import dataclass
from urllib.parse import urlparse
import calendar
from spacy.lang.ru.stop_words import STOP_WORDS as spacy_stopwords  # Стоп-слова из spaCy

In [ ]:
client = openai.OpenAI(api_key="", base_url="")
response = client.chat.completions.create(
    model="gpt-4o-mini-continue",
    messages=[{"role": "user", "content": "Привет! Как дела?"}]
)

print(response.choices[0].message.content)

Привет! У меня все хорошо, спасибо! А как у тебя дела?


In [ ]:
# 1. Подключение Google Drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Магистратура Итмо"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


РАБОТА С JSON

In [ ]:
@dataclass
class BaseEvent:
    id: int
    batch_id: int
    user_id: str
    timestamp: str
    event_type: str
    record_id: str
    related_file: str
    log_record_counter: int
    event_context: str
    environment: str

    @staticmethod
    def from_dict(obj: Any) -> 'BaseEvent':
        _id = int(obj.get("id"))
        _batch_id = int(obj.get("batch_id"))
        _user_id = str(obj.get("user_id"))
        _timestamp = str(obj.get("timestamp"))
        _event_type = str(obj.get("event_type"))
        _record_id = str(obj.get("record_id"))
        _related_file = str(obj.get("related_file"))
        _log_record_counter = int(obj.get("log_record_counter"))
        _event_context = str(obj.get("event_context"))
        _environment = str(obj.get("environment"))
        return BaseEvent(_id, _batch_id, _user_id, _timestamp, _event_type, _record_id, _related_file, _log_record_counter, _event_context, _environment)

@dataclass
class AudioEvent:  # Добавил класс AudioEvent, так как он используется в Root
     # Нужно определить поля AudioEvent, если они вам нужны.  Если нет, можно оставить пустым.
     id: int #пример
     @staticmethod
     def from_dict(obj: Any) -> 'AudioEvent':
         _id = int(obj.get("id")) # пример
         return AudioEvent(_id)


@dataclass
class Root:
    base_events: List[BaseEvent]
    audio_events: List[AudioEvent]

    @staticmethod
    def from_dict(obj: Any) -> 'Root':
        _base_events = [BaseEvent.from_dict(y) for y in obj.get("base_events")]
        _audio_events = [AudioEvent.from_dict(y) for y in obj.get("audio_events", [])]  # Исправлено: используем AudioEvent
        return Root(_base_events, _audio_events)



# 3. Функция для рекурсивного поиска JSON файлов
def find_json_files(root_path: str) -> List[str]:
    """
    Рекурсивно находит все JSON файлы в заданной директории и её поддиректориях,
    соответствующие шаблону "batch-*".

    Args:
        root_path: Путь к корневой директории для поиска.

    Returns:
        Список путей к найденным JSON файлам.
    """
    json_files = []
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Проверяем, соответствует ли текущая директория шаблону "batch-*"
        if os.path.basename(dirpath).startswith("batch-"):
            for filename in filenames:
                if filename.endswith(".json"):
                    json_files.append(os.path.join(dirpath, filename))
    return json_files


# 4. Основная логика обработки
def process_json_files(base_path: str):
    """
    Находит все JSON файлы в подпапках "batch-*" внутри заданной директории,
    загружает их и обрабатывает с использованием классов BaseEvent и Root.

    Args:
        base_path:  Базовый путь к папке "EventLogger" на Google Диске.
    """

    all_roots = []  # Список для хранения всех объектов Root

    # Формируем полный путь к папке EventLogger
    event_logger_path = os.path.join(base_path, "Manuspect", "logs", "EventLogger")

    # Находим все JSON файлы
    json_file_paths = find_json_files(event_logger_path)

    if not json_file_paths:
        print("JSON файлы не найдены.")
        return

    # Обрабатываем каждый JSON файл
    for json_file_path in json_file_paths:
        try:
            with open(json_file_path, 'r') as f:
                json_content = f.read()
                json_data = json.loads(json_content)  # Загружаем JSON данные
                root = Root.from_dict(json_data)       # Создаем объект Root
                all_roots.append(root)                 # Добавляем в список
                print(f"Файл {json_file_path} успешно обработан.")

        except (json.JSONDecodeError, FileNotFoundError, KeyError, TypeError) as e:
            print(f"Ошибка при обработке файла {json_file_path}: {e}")
        except Exception as e:
             print(f"Неожиданная ошибка при обработке {json_file_path}: {e}")

    #  all_roots теперь содержит список всех объектов Root, полученных из всех JSON файлов.
    #  Дальше можно с ними работать, например, анализировать данные.
    print(f"\nВсего обработано объектов Root: {len(all_roots)}")
    return all_roots

# 5. Запуск обработки
# Укажите базовый путь к папке "Магистратура Итмо" на вашем Google Диске
all_roots = process_json_files(base_path)

# Пример доступа к данным (после того, как process_json_files отработала)
if all_roots:
    first_root = all_roots[0]
    print(f"Количество BaseEvent в первом файле: {len(first_root.base_events)}")
    if first_root.base_events:
        first_event = first_root.base_events[0]
        print(f"ID первого события: {first_event.id}")
        print(f"user_id первого события: {first_event.user_id}")

Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-403/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-413/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-412/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-406/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-404/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-408/events.json успешно обработан.
Файл /content/drive/MyDrive/Магистратура Итмо/Manuspect/logs/EventLogger/01JKD9VP1Z8Z8H4S98797XEJD5/batch-411/events.json успешно обработан.
Файл /content

In [ ]:
def create_dataframe(all_roots: List[object]) -> pd.DataFrame:
    """
    Создает DataFrame из списка объектов Root, извлекая данные из BaseEvent.

    Args:
        all_roots: Список объектов Root, полученных из JSON файлов.

    Returns:
        pd.DataFrame: DataFrame, содержащий данные из BaseEvent.  Если данных нет,
                      возвращается пустой DataFrame.
    """

    all_events_data = []  # Список для хранения данных всех событий

    for root in all_roots:
        for base_event in root.base_events:
            event_data = {
                'id': base_event.id,
                'batch_id': base_event.batch_id,
                'user_id': base_event.user_id,
                'timestamp': base_event.timestamp,
                'event_type': base_event.event_type,
                'record_id': base_event.record_id,
                'related_file': base_event.related_file,
                'log_record_counter': base_event.log_record_counter,
                'event_context': base_event.event_context,
                'environment': base_event.environment,
            }
            all_events_data.append(event_data)

    if not all_events_data:
        print("Нет данных для создания DataFrame.")
        return pd.DataFrame()  # Возвращаем пустой DataFrame

    df = pd.DataFrame(all_events_data)
    return df

# 1. Получаем DataFrame (предполагается, что all_roots уже определен)
if 'all_roots' in locals() and all_roots: # Проверяем, определена ли переменная и не пуста ли она
    events_df = create_dataframe(all_roots)

ПРЕДОБРАБОТКА ДАТАСЕТА

In [ ]:
# Список возможных разделителей
SEPARATORS = ['::', ' - ', ' | ', ' — ', ' – ']  # Можно добавить другие разделители

# Регулярные выражения для выявления URL
URL_PATTERN = r'https?://[^\s]+|www\.[^\s]+|[^\s]+\.(com|ru|org|net|edu|gov|io)[^\s]*'

# Список известных браузеров (по classname, process_path или program_title)
BROWSER_INDICATORS = {
    'Chrome_WidgetWin_1': 'Google Chrome',
    'MozillaWindowClass': 'Firefox',
    'Edge': 'Microsoft Edge',
    # Можно добавить другие браузеры
}

# Список названий браузеров для удаления из tab_title
BROWSER_NAMES = [
    'Google Chrome', 'Chrome', 'Firefox', 'Mozilla Firefox',
    'Microsoft Edge', 'Edge', 'Safari', 'Opera'
]

# Функция для очистки и сокращения URL
def clean_and_shorten_url(url):
    """
    Очищает и сокращает URL, оставляя только домен и основную часть пути.

    Args:
        url: Исходная URL-строка.

    Returns:
        Сокращенная URL-строка.
    """
    if not url:
        return ''

    try:
        parsed_url = urlparse(url)
        domain = parsed_url.netloc
        path = parsed_url.path
        if domain and path:
            return f"{domain}{path[:50]}"  # Ограничиваем длину пути
        return domain or path
    except:
        return url

# Функция для очистки строки от URL и сокращения
def clean_and_shorten(text, is_url=False):
    """
    Очищает строку от URL и сокращает её, удаляя лишние детали.

    Args:
        text: Исходная строка.
        is_url: Флаг, указывающий, является ли строка URL.

    Returns:
        Очищенная и сокращенная строка.
    """
    if not text:
        return ''

    if is_url:
        return clean_and_shorten_url(text)

    # Удаляем URL
    cleaned_text = re.sub(URL_PATTERN, '', text, flags=re.IGNORECASE)

    # Удаляем параметры URL (например, ?param=value)
    cleaned_text = re.sub(r'\?.*$', '', cleaned_text)

    # Удаляем лишние пробелы
    cleaned_text = cleaned_text.strip()

    # Сокращаем длинные пути (оставляем только последние 2 части)
    if '\\' in cleaned_text or '/' in cleaned_text:
        parts = cleaned_text.replace('\\', '/').split('/')
        if len(parts) > 2:
            cleaned_text = '/'.join(parts[-2:])

    return cleaned_text

# Функция для удаления названий браузеров из строки
def remove_browser_names(text, browser_names):
    """
    Удаляет названия браузеров из строки.

    Args:
        text: Исходная строка.
        browser_names: Список названий браузеров.

    Returns:
        Строка без названий браузеров.
    """
    if not text:
        return ''

    cleaned_text = text
    for browser in browser_names:
        # Удаляем browser из конца строки
        pattern = rf'\s*-\s*{re.escape(browser)}$'
        cleaned_text = re.sub(pattern, '', cleaned_text, flags=re.IGNORECASE)
        # Удаляем browser из середины строки
        pattern = rf'\s*-\s*{re.escape(browser)}\s*-\s*'
        cleaned_text = re.sub(pattern, ' - ', cleaned_text, flags=re.IGNORECASE)

    return cleaned_text.strip()

# Функция для извлечения информации из environment
def extract_environment_info(environment_str):
    """
    Извлекает информацию из строки environment, разделяя program_title на корневое приложение и вкладки,
    а также извлекая дополнительные поля (mouse_x, mouse_y, modifiers, log_windows).

    Args:
        environment_str: Строка в формате JSON, представляющая environment.

    Returns:
        Список словарей с извлеченной информацией или None, если не удалось извлечь.
    """
    if not isinstance(environment_str, str):
        return None

    try:
        environment_data = json.loads(environment_str)
        results = []
        if isinstance(environment_data, dict) and 'log_windows' in environment_data:
            for window in environment_data['log_windows']:
                if isinstance(window, dict):
                    # Извлекаем program_title, classname и process_path
                    program_title = window.get('program_title', '').strip()
                    classname = window.get('classname', '').strip()
                    process_path = window.get('process_path', '').strip()

                    # Инициализируем значения
                    root_app = ''
                    tab_title = ''

                    # Проверяем, является ли это браузером
                    is_browser = False
                    for indicator, app_name in BROWSER_INDICATORS.items():
                        if indicator in classname or indicator.lower() in process_path.lower():
                            is_browser = True
                            root_app = app_name
                            break

                    if is_browser:
                        # Для браузеров program_title - это URL или вкладка
                        # Удаляем название браузера из program_title
                        cleaned_title = remove_browser_names(program_title, BROWSER_NAMES)
                        tab_title = clean_and_shorten(cleaned_title, is_url=True)
                    else:
                        # Проверяем, является ли program_title путем к файлу
                        if '\\' in program_title or '/' in program_title:
                            # Извлекаем полный путь до папки и имя файла
                            file_name = os.path.basename(program_title)
                            # Удаляем расширение .exe, .bat, .dll и другие
                            specific_part = re.sub(r'\.(exe|bat|dll)$', '', file_name, flags=re.IGNORECASE).strip()

                            # Извлекаем путь до папки
                            dir_path = os.path.dirname(program_title)

                            # Устанавливаем root_app как путь до папки
                            root_app = dir_path

                            # Устанавливаем tab_title как имя файла
                            tab_title = f"\\{specific_part}" if specific_part else ''
                        else:
                            # Проверяем наличие разделителей
                            last_separator = None
                            last_pos = -1

                            # Ищем последний разделитель в строке
                            for sep in SEPARATORS:
                                pos = program_title.rfind(sep)
                                if pos > last_pos:
                                    last_pos = pos
                                    last_separator = sep

                            if last_separator and last_pos > 0:
                                # Разделяем по последнему найденному разделителю
                                tab_title = program_title[:last_pos].strip()
                                root_app = program_title[last_pos + len(last_separator):].strip()

                                # Очищаем root_app и tab_title
                                root_app = clean_and_shorten(root_app)
                                tab_title = clean_and_shorten(tab_title)
                            else:
                                # Если разделителей нет, вся строка - root_app
                                root_app = clean_and_shorten(program_title)

                    # Очищаем process_path
                    process_path = clean_and_shorten(process_path)

                    # Извлекаем дополнительные поля
                    result = {
                        'program_title': program_title,
                        'root_app': root_app,
                        'tab_title': tab_title,
                        'classname': classname,
                        'process_path': process_path,
                        'is_active': window.get('is_active', False),
                        'z_index': window.get('z_index', 0),
                        'mouse_x': environment_data.get('mouse_x', None),
                        'mouse_y': environment_data.get('mouse_y', None),
                        'modifiers': environment_data.get('modifiers', None),
                        'window_left': window.get('window_left', None),
                        'window_top': window.get('window_top', None),
                        'window_right': window.get('window_right', None),
                        'window_bottom': window.get('window_bottom', None)
                    }
                    results.append(result)
        return results

    except (json.JSONDecodeError, TypeError):
        return None

# Загружаем данные (предполагаем, что events_df уже загружен)
data = events_df

# Проверяем наличие столбца 'environment'
if 'environment' not in data.columns:
    print("Столбец 'environment' не найден в данных.")
    exit()

# Применяем функцию extract_environment_info к столбцу 'environment'
data['env_info'] = data['environment'].apply(extract_environment_info)

# Удаляем строки, где не удалось извлечь информацию
data = data[data['env_info'].notna()]

# Разворачиваем списки env_info в отдельные строки
data = data.explode('env_info')

# Создаем новые столбцы из словарей env_info
data = pd.concat([data.drop(['env_info'], axis=1),
data['env_info'].apply(pd.Series)], axis=1)

# Удаляем строки с пустыми значениями root_app
data = data[data['root_app'].notna() & (data['root_app'] != '')]

# Удаляем ненужные столбцы
columns_to_drop = ['batch_id', 'related_file', 'log_record_counter', 'program_title', 'program_titles']
data = data.drop(columns=columns_to_drop, errors='ignore')

# Выводим информацию о корневых приложениях
print("\nУникальные корневые приложения (root_app) и их количество:")
print(data['root_app'].value_counts())

# Выводим информацию о вкладках (tab_title)
print("\nУникальные вкладки (tab_title) и их количество:")
print(data['tab_title'].value_counts())


Уникальные корневые приложения (root_app) и их количество:
root_app
Google Chrome                                 9353
Release                                         60
EventLogger                                     45
logs                                            30
01JK6CGKAHCZRRG8N5WPEGZF9M                      29
Создание фрагмента экрана                       29
Cisco Secure Client                             28
batch-450                                       26
Блокнот                                         22
Зашифровать                                     22
tmp                                             14
01JKDAJNV3EJE59A3E64Q76AJH                      14
01JKD53CB9C94YFB17Q38KNTHV                      10
99⁺                                              9
Загрузки                                         8
desktop-app-build-main (5)                       8
КриптоАРМ                                        6
демо1_Ivanova P.D.zip (пробная копия)            6
Открытие     

In [ ]:
def save_to_csv(df: pd.DataFrame, filename: str, base_path: str):
    """
    Сохраняет DataFrame в CSV файл на Google Диске.

    Args:
        df: DataFrame для сохранения.
        filename: Имя файла для сохранения.
        base_path: Путь к папке на Google Диске, куда нужно сохранить файл.
    """
    full_path = os.path.join(base_path, filename)  # Полный путь к файлу
    try:
        df.to_csv(full_path, index=False)  # index=False чтобы не сохранять индексы строк
        print(f"DataFrame успешно сохранен в файл: {full_path}")
    except Exception as e:
        print(f"Ошибка при сохранении файла: {e}")

DataFrame успешно сохранен в файл: /content/drive/MyDrive/Магистратура Итмо/data2.csv


In [ ]:
save_to_csv(data, "data2.csv", base_path)

КЛАССИФИКАЦИЯ ПО КОНТЕКСТАМ

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Подключение Google Drive (если еще не подключен)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

def load_csv_from_drive(filename: str, base_path: str, column_separators: dict = None, default_sep: str = '\t', default_encoding: str = 'utf-8') -> pd.DataFrame:
    """Loads a CSV file from Google Drive with different separators."""
    full_path = os.path.join(base_path, filename)
    logging.info(f"Загрузка файла: {full_path}")
    try:
        sep = default_sep
        if filename == "df_context.csv":
            sep = ';'
        elif filename == "data2.csv":
            sep = ','
        elif filename == "classified_user_data26.csv":
            sep = ','  # Указываем разделитель для вашего файла

        encoding = default_encoding

        try:
            # Сначала читаем заголовки
            headers = pd.read_csv(full_path, sep=sep, encoding=encoding, nrows=0).columns
        except UnicodeDecodeError:
            headers = pd.read_csv(full_path, sep=sep, encoding='cp1251', nrows=0).columns
        logging.info(f"Заголовки: {headers}")

        converters = {}
        if column_separators:
            for col, col_sep in column_separators.items():
                if col in headers:
                    converters[col] = lambda x: x.split(col_sep)
                else:
                    logging.warning(f"Предупреждение: Столбец '{col}' не найден в файле '{filename}'.  Преобразование для него пропущено.")

        try:
            df = pd.read_csv(full_path, sep=sep, encoding=encoding, converters=converters)
        except UnicodeDecodeError:
            df = pd.read_csv(full_path, sep=sep, encoding='cp1251', converters=converters)


        logging.info(f"File {full_path} successfully loaded.")
        logging.info(f"Columns in {filename}: {df.columns.tolist()}")
        print(f"Первые 5 строк:\n{df.head()}")  # Для отладки
        return df

    except FileNotFoundError:
        logging.error(f"File {full_path} not found.")
        return None
    except Exception as e:
        logging.error(f"Error loading file {full_path}: {e}")
        return None

# --- Тестирование ---
base_path = '/content/drive/MyDrive/Магистратура Итмо'  # Замените на ваш путь!
# base_path = './'  # Для локального запуска

df_context = load_csv_from_drive('df_context.csv', base_path, column_separators={'context': '|'})
processed_data = load_csv_from_drive('data2.csv', base_path)

if df_context is not None:
  print("df_context loaded successfully")

if processed_data is not None:
  print("processed_data loaded successfully")

Первые 5 строк:
           tab_title             context    tags
0    Adobe Photoshop  [Дизайн и креатив]  Design
1  Adobe Illustrator  [Дизайн и креатив]  Design
2          CorelDRAW  [Дизайн и креатив]  Design
3  Affinity Designer  [Дизайн и креатив]  Design
4               GIMP  [Дизайн и креатив]  Design
Первые 5 строк:
    id                               user_id                      timestamp  \
0  405  43db9537-0b46-47a1-937f-6d51af423cfe  2025-02-06T12:36:07.186+03:00   
1  406  43db9537-0b46-47a1-937f-6d51af423cfe  2025-02-06T12:36:07.577+03:00   
2  407  43db9537-0b46-47a1-937f-6d51af423cfe  2025-02-06T12:36:07.686+03:00   
3  408  43db9537-0b46-47a1-937f-6d51af423cfe  2025-02-06T12:36:07.764+03:00   
4  409  43db9537-0b46-47a1-937f-6d51af423cfe  2025-02-06T12:36:09.018+03:00   

  event_type                   record_id event_context  \
0     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5           NaN   
1     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5           NaN   
2     VISUAL  01JKD9VP1Z8

In [ ]:
DEFAULT_CATEGORY = "Операционная деятельность"
VALID_CATEGORIES = [
    'Встречи и коммуникации', 'Дизайн и креатив',
    'Обучение и саморазвитие', 'Производительность и организация',
    'Развлечения и медиа', 'Разработка', 'Финансы и бухгалтерия',
    'Электронная коммерция и покупки', 'Операционная деятельность'
]

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Загружаем модель spaCy (ru_core_news_sm - маленькая, быстрая; ru_core_news_md - средняя; ru_core_news_lg - большая, качественная)
try:
    nlp = spacy.load("ru_core_news_sm")  # Попробуем маленькую модель
    print("Модель spaCy (ru_core_news_sm) успешно загружена.")
except OSError:
    try:
        # Если маленькая не загрузилась, пробуем среднюю
        nlp = spacy.load("ru_core_news_md")
        print("Модель spaCy (ru_core_news_md) успешно загружена.")
    except OSError:
        try:
            # Если средняя не загрузилась, пробуем большую
            nlp = spacy.load("ru_core_news_lg")
            print("Модель spaCy (ru_core_news_lg) успешно загружена.")
        except OSError:
            print("Ошибка: не удалось загрузить ни одну модель spaCy для русского языка.")
            print("Убедитесь, что у вас установлены spaCy и хотя бы одна русская модель.")
            print("Выполните в терминале:")
            print("  pip install spacy")
            print("  python -m spacy download ru_core_news_sm")
            print("  (или ru_core_news_md, ru_core_news_lg)")
            raise  # Завершаем выполнение, если ни одна модель не загрузилась


def preprocess_text(text: str) -> str:
    """
    Preprocesses text for TF-IDF:
    1. Lowercases
    2. Removes punctuation
    3. Removes numbers
    4. Tokenizes (using razdel)
    5. Removes stopwords (using spaCy stopwords)
    6. Lemmatizes (using spaCy)
    """
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenize (using razdel)
    words = [token.text for token in tokenize(text)]

    # Remove stopwords (using spaCy stopwords)
    # stop_words = set(stopwords.words('russian'))  # Заменяем на стоп-слова spaCy
    words = [word for word in words if word not in spacy_stopwords]

    # Lemmatize (using spaCy)
    # Объединяем слова обратно в строку ПЕРЕД лемматизацией spaCy
    text = " ".join(words)
    doc = nlp(text)
    words = [token.lemma_ for token in doc]  # Используем атрибут lemma_

    return " ".join(words)

Модель spaCy (ru_core_news_sm) успешно загружена.


In [ ]:
def create_tfidf_model(training_df: pd.DataFrame) -> Tuple[TfidfVectorizer, pd.DataFrame, pd.DataFrame]:
    """
    Создает и обучает TF-IDF модель. Возвращает векторизатор, векторизованные данные и *копию* DataFrame с обработанными заголовками.
    """
    if 'tab_title' not in training_df.columns or 'context' not in training_df.columns:
        raise ValueError("Training data must have 'tab_title' and 'context' columns.")

    # ВАЖНО: Работаем с копией, чтобы не изменять исходный DataFrame
    training_df_copy = training_df.copy()
    training_df_copy['processed_title'] = training_df_copy['tab_title'].apply(preprocess_text)

    try:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(training_df_copy['processed_title'])
        tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
        return vectorizer, tfidf_df, training_df_copy  # Возвращаем измененную копию!
    except Exception as e:
        logging.error(f"Error in create_tfidf_model: {e}")
        return None, None, None

In [ ]:
def select_few_shot_examples(tab_title: str, training_df: pd.DataFrame, tfidf_vectorizer: TfidfVectorizer, tfidf_df: pd.DataFrame, n_examples: int = 3) -> str:
    """
    Выбирает наиболее релевантные примеры (few-shot) с использованием TF-IDF и косинусного сходства.
    """
    processed_title = preprocess_text(tab_title)
    title_vector = tfidf_vectorizer.transform([processed_title])
    cosine_similarities = cosine_similarity(title_vector, tfidf_vectorizer.transform(training_df['processed_title']))
    most_similar_indices = cosine_similarities.argsort(axis=None)[::-1][:n_examples]
    examples = []
    for i in most_similar_indices:
        title = training_df['tab_title'].iloc[i]
        context = training_df['context'].iloc[i]
        examples.append(f"'{title}': {context}")
    return "\n".join(examples)

In [ ]:
def normalize_title(title):
    """Нормализует заголовок вкладки."""
    if isinstance(title, float):  # Обрабатываем случай, когда title является числом (например, NaN)
        return ''
    title = str(title).strip()  # Преобразуем в строку и удаляем начальные/конечные пробелы
    title = re.sub(r'[^\w\s\-а-яА-ЯёЁ]', '', title, flags=re.UNICODE).lower()  # Удаляем пунктуацию, оставляем только буквы, цифры, пробелы, дефисы и русские буквы
    title = re.sub(r'\s+', ' ', title).strip() # Заменяем множественные пробелы на один
    title = re.sub(r'^[-—\s]+|[-—\s]+$', '', title)  # Удаляем ведущие и замыкающие дефисы и пробельные символы
    return title

In [ ]:
def create_word_to_context_mapping(training_df: pd.DataFrame) -> Dict[str, List[str]]:
    """Создает отображение слов из заголовков в контексты."""
    word_to_context = {}
    if 'tab_title' in training_df.columns and 'context' in training_df.columns:
        for _, row in training_df.iterrows():
            if pd.notna(row['tab_title']) and str(row['tab_title']).strip() != "":
                normalized_title = normalize_title(row['tab_title'])
                words = normalized_title.split()
                for word in words:
                    if word not in word_to_context:
                        word_to_context[word] = []
                    if pd.notna(row['context']) and row['context'] not in word_to_context[word]:
                        word_to_context[word].append(row['context'])

    else:
        logging.warning("Внимание: Столбец 'tab_title' или 'context' не найден в обучающих данных.")
    return word_to_context

In [ ]:
def create_title_to_category_mapping(df: pd.DataFrame) -> Dict[str, str]:
    """Создает словарь соответствия заголовков и категорий (для постобработки)."""
    title_to_category = {}  # Словарь для хранения соответствий
    # Проверяем, что DataFrame не None и содержит нужные столбцы
    if df is not None and 'tab_title' in df.columns and 'category' in df.columns:
        for _, row in df.iterrows():
            title = row['tab_title']
            #  Проверяем, что title является строкой
            if isinstance(title, str):
                normalized = normalize_title(title)  # Нормализуем заголовок
                # Проверяем, что нормализованный заголовок не пустой
                if normalized:
                    title_to_category[normalized] = row['category']  # Добавляем соответствие в словарь
    return title_to_category

In [ ]:
def lru_cache_with_time_limit(maxsize=128, typed=False, time_limit_seconds=3600):
    """Декоратор для кэширования с ограничением по времени."""
    def wrapper_cache(func):
        @functools.lru_cache(maxsize=maxsize, typed=typed)
        def cached_func(*args, **kwargs):
            return func(*args, **kwargs)

        cached_func.lifetime = time_limit_seconds
        cached_func.expiration = time.monotonic() + cached_func.lifetime

        @functools.wraps(func)
        def wrapped_func(*args, **kwargs):
            if time.monotonic() >= cached_func.expiration:
                cached_func.cache_clear()
                cached_func.expiration = time.monotonic() + cached_func.lifetime
            return cached_func(*args, **kwargs)
        return wrapped_func
    return wrapper_cache

In [ ]:
@lru_cache_with_time_limit(maxsize=None, time_limit_seconds=86400)
def classify_with_chatgpt(tab_title: str, model="gpt-4o-mini-continue", context_str: str = "", client=None, title_to_category_serialized: str = None) -> str:
    """Классифицирует текст с помощью ChatGPT, с постобработкой."""
    try:
        title_to_category = json.loads(title_to_category_serialized) if title_to_category_serialized else None

        system_message = (
            "You are a helpful assistant that categorizes user activities based on web browser titles and application names. "
            "Return ONLY one of the following category names: 'Встречи и коммуникации', 'Дизайн и креатив', 'Обучение и саморазвитие', "
            "'Производительность и организация', 'Развлечения и медиа', 'Разработка', 'Финансы и бухгалтерия', 'Электронная коммерция и покупки', 'Операционная деятельность'. "
            "If you cannot determine the category, return 'Операционная деятельность'. "

        )
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": f"Title: {tab_title}"}
        ]
        if context_str:
            messages.insert(1, {"role": "system", "content": f"Here are some known title-category mappings for additional context:\n{context_str}"})

        response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        result = response.choices[0].message.content.strip()

        # --- Постобработка (как и раньше) ---
        if title_to_category:
            normalized_title = normalize_title(tab_title)
            for original_title, category in title_to_category.items():
                if normalize_title(original_title) == normalized_title:
                    result = category
                    logging.info(f"Постобработка: '{tab_title}' -> '{result}' (из словаря)")
                    break
            else:
                if result not in VALID_CATEGORIES:
                    logging.info(f"Постобработка: '{result}' не является допустимой категорией.")
                    result = DEFAULT_CATEGORY
        return result

    except Exception as e:
        logging.error(f"Ошибка API ChatGPT: {e}")
        if hasattr(e, 'response'):
            logging.error(f"Ответ от API: {e.response}")
        return DEFAULT_CATEGORY

In [ ]:
def classify_user_activity(user_df: pd.DataFrame, training_df: pd.DataFrame, use_chatgpt=False, chatgpt_model="gpt-4o-mini-continue", client=None) -> pd.DataFrame:
    """Классифицирует активность пользователя."""

    # Обработка пустого user_df
    if user_df.empty:
        logging.warning("Входной DataFrame пользователя пуст. Возвращается пустой DataFrame.")
        return pd.DataFrame()

    try:
        title_to_context = {}
        print(f"title_to_context: {title_to_context}")
        if 'tab_title' in training_df.columns and 'context' in training_df.columns:
            title_to_context = {
                normalize_title(row['tab_title']): row['context']
                for _, row in training_df.iterrows()
                if pd.notna(row['tab_title']) and str(row['tab_title']).strip() != ""
            }
        else:
            logging.warning("Внимание: Столбец 'tab_title' или 'context' не найден в обучающих данных.")

        context_str = "\n".join([f"'{title}': {context}" for title, context in title_to_context.items()])
        word_to_context = create_word_to_context_mapping(training_df)
        chatgpt_cache = {}  # Кэш для ChatGPT

        def classify_row(row):
            tab_title_col = 'tab_title'
            root_app_col = 'root_app'

            if tab_title_col not in row:
                logging.warning(f"Warning: '{tab_title_col}' not found in row.  Using empty string.")
                tab_title = ''
            else:
                tab_title = row[tab_title_col]

            if pd.isna(tab_title) or str(tab_title).strip() == '':
                logging.info("  Обрабатывается пустой tab_title. Используется категория по умолчанию.")
                return DEFAULT_CATEGORY, {} # Возвращаем пустой словарь scores

            normalized_tab_title = normalize_title(tab_title)
            print(f"Нормализованный заголовок: {normalized_tab_title}") # <--- ВОТ ЭТА СТРОКА

            if root_app_col not in row:
                logging.warning(f"Warning: '{root_app_col}' not found in row. Using empty list.")
                root_app_list = []
            else:
                root_app_list = row[root_app_col]

            if isinstance(root_app_list, str):
                root_app_list = [root_app_list]
            elif not isinstance(root_app_list, list):
                root_app_list = []
            root_app_str = ', '.join(root_app_list)

            category_scores: Dict[str, float] = {}

            # 1. Точное сопоставление
            if normalized_tab_title in title_to_context:
                category_scores[title_to_context[normalized_tab_title]] = category_scores.get(title_to_context[normalized_tab_title], 0) + 10

            # 2. ChatGPT
            if use_chatgpt:
                cache_key = (normalized_tab_title, root_app_str)
                if cache_key in chatgpt_cache:
                    category = chatgpt_cache[cache_key]
                else:
                    title_to_category_serialized = json.dumps(title_to_category_mapping)
                    category = classify_with_chatgpt(tab_title, model=chatgpt_model, context_str=context_str, client=client, title_to_category_serialized=title_to_category_serialized)
                    if category in VALID_CATEGORIES:
                        chatgpt_cache[cache_key] = category

                if category in VALID_CATEGORIES:
                    category_scores[category] = category_scores.get(category, 0) + 5
                else:
                    category_scores[category] = category_scores.get(category, 0) + 2

            # 3. Пословное сопоставление
            words = normalized_tab_title.split()
            for word in words:
                if word in word_to_context:
                    for context in word_to_context[word]:
                        category_scores[context] = category_scores.get(context, 0) + 6

            if category_scores:
                best_category = max(category_scores, key=category_scores.get)
                return best_category, category_scores # Возвращаем категорию и scores
            else:
                return DEFAULT_CATEGORY, {} # Возвращаем категорию по умолчанию и пустой словарь

            # Штраф за отнесение к дефолтной категории, если другие категории получили баллы
            if category_scores and DEFAULT_CATEGORY in category_scores and any(c != DEFAULT_CATEGORY for c in category_scores):
                category_scores[DEFAULT_CATEGORY] -= 3  # Штраф

            if category_scores:
                best_category = max(category_scores, key=category_scores.get)
                return best_category, category_scores # Возвращаем категорию и scores
            else:
                return DEFAULT_CATEGORY, {} # Возвращаем категорию по умолчанию и пустой словарь

        # Собираем результаты классификации и scores
        results = user_df.apply(classify_row, axis=1, result_type='expand')
        user_df['predicted_context'] = results[0]
        user_df['normalized_tab_title'] = user_df['tab_title'].apply(normalize_title)
        user_df['category_scores'] = results[1] # Добавляем столбец category_scores
        return user_df

    except Exception as e:
        logging.error(f"Произошла ошибка в classify_user_activity: {e}")
        return pd.DataFrame()
title_to_category_mapping = create_title_to_category_mapping(df_context)

In [ ]:
# --- Основной код ---
base_path = "/content/drive/MyDrive/Магистратура Итмо"
training_filename = "df_context.csv"
user_filename = "data2.csv"
training_data = load_csv_from_drive(training_filename, base_path)
user_data = load_csv_from_drive(user_filename, base_path, column_separators={'root_app': ';'})


if training_data is None or user_data is None:
    logging.error("Ошибка загрузки данных. Проверьте пути к файлам и их содержимое.")
else:
    use_chatgpt = True  # Включите, чтобы использовать ChatGPT
    chatgpt_model = "gpt-4o-mini-continue" #Изменили на 3.5

    classified_data = classify_user_activity(user_data, training_data, use_chatgpt, chatgpt_model, client=client) # Передаем client

    required_columns = ['tab_title', 'root_app', 'predicted_context']
    if all(col in classified_data.columns for col in required_columns):
        print(classified_data[required_columns].head(20))
    else:
        missing_cols = [col for col in required_columns if col not in classified_data.columns]
        logging.error(f"Ошибка: Отсутствуют следующие столбцы: {', '.join(missing_cols)}")
        logging.error(f"Столбцы в classified_data: {classified_data.columns}")

    output_filename = "classified_user_data26.csv"  # Новое имя файла
    output_path = os.path.join(base_path, output_filename)
    classified_data.to_csv(output_path, index=False, sep=';', encoding='utf-8')
    logging.info(f"Данные сохранены в: {output_path}")

Выходные данные были обрезаны до нескольких последних строк (5000).
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: статистика
Нормализованный заголовок: статистика
Нормализованный заголовок: диалог 240671322481360896
Нормализованный заголовок: диалог 240671322481360896
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: обновить список номеров
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: telegram web
Нормализованный заголовок: робот исход - дубли - тест
Нормализованный заголовок: telegram web
Нормализованный заголовок: telegram web


ВИЗУАЛИЗАЦИЯ

In [ ]:
def preprocess_data(data):
    """Предварительная обработка данных (один раз)."""
    if data['timestamp'].dtype != 'datetime64[ns]':
        data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
        data.dropna(subset=['timestamp'], inplace=True)
    data['date'] = data['timestamp'].dt.date
    data['weekday'] = data['timestamp'].dt.day_name()  # Оставляем, но не используем в группировке
    return data

In [ ]:
def calculate_durations(data):  # Убираем group_by_weekday
    """Рассчитывает длительности (после фильтрации)."""
    if data.empty:
        print("Нет данных для выбранных фильтров.")
        return None, None, None, None

    group_cols_root_app = ['root_app', 'date']
    if 'user_id' in data.columns:
        group_cols_root_app.insert(0, 'user_id')
    # Убираем вставку 'weekday'
    grouped_root_app = data.groupby(group_cols_root_app)['timestamp'].agg(['min', 'max', 'count']).reset_index()
    grouped_root_app['duration_seconds'] = (grouped_root_app['max'] - grouped_root_app['min']).dt.total_seconds()
    grouped_root_app['duration_minutes'] = grouped_root_app['duration_seconds'] / 60

    group_cols_tab_title = ['tab_title', 'date']
    if 'user_id' in data.columns:
        group_cols_tab_title.insert(0, 'user_id')
    # Убираем вставку 'weekday'
    grouped_tab_title = data.groupby(group_cols_tab_title)['timestamp'].agg(['min', 'max', 'count']).reset_index()
    grouped_tab_title['duration_seconds'] = (grouped_tab_title['max'] - grouped_tab_title['min']).dt.total_seconds()
    grouped_tab_title['duration_minutes'] = grouped_tab_title['duration_seconds'] / 60

    group_cols_tab_title_per_app = ['root_app', 'tab_title', 'date']
    if 'user_id' in data.columns:
        group_cols_tab_title_per_app.insert(0, 'user_id')
    # Убираем вставку 'weekday'
    grouped_tab_title_per_app = data.groupby(group_cols_tab_title_per_app)['timestamp'].agg(['min', 'max', 'count']).reset_index()
    grouped_tab_title_per_app['duration_seconds'] = (grouped_tab_title_per_app['max'] - grouped_tab_title_per_app['min']).dt.total_seconds()
    grouped_tab_title_per_app['duration_minutes'] = grouped_tab_title_per_app['duration_seconds'] / 60

    def extract_context_durations(row):
        try:
            return ast.literal_eval(row['category_scores']) if isinstance(row['category_scores'], str) else json.loads(row['category_scores'])
        except (ValueError, SyntaxError, json.JSONDecodeError):
            return {}

    context_df = data.apply(extract_context_durations, axis=1).apply(pd.Series)
    context_df = context_df.rename(columns={col: f"context_{col}" for col in context_df.columns})
    data = pd.concat([data.drop(columns=['category_scores']), context_df], axis=1)
    context_cols = [col for col in data.columns if col.startswith('context_')]
    for col in context_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

    group_cols_context = ['date']
    if 'user_id' in data.columns:
        group_cols_context.insert(0, 'user_id')
    # Убираем вставку 'weekday'
    context_duration_df = data.groupby(group_cols_context)[context_cols].sum().reset_index()
    for col in context_cols:
        context_duration_df[col] = context_duration_df[col] / 60

    return grouped_root_app, grouped_tab_title, context_duration_df, grouped_tab_title_per_app

In [ ]:
def visualize_durations_refined(grouped_root_app, grouped_tab_title, context_duration_df, grouped_tab_title_per_app):
    """Визуализирует данные."""
    if grouped_root_app is not None and not grouped_root_app.empty:
        grouped_root_app = grouped_root_app.sort_values('duration_minutes', ascending=False)
        grouped_root_app['duration_formatted'] = grouped_root_app['duration_minutes'].apply(lambda x: f"{int(x // 60)}:{int(x % 60):02d}")
        plt.figure(figsize=(14, 7))
        sns.barplot(x='duration_minutes', y='root_app', data=grouped_root_app, orient='h', palette="viridis", errorbar=None)
        for i, (value, label) in enumerate(zip(grouped_root_app['duration_minutes'], grouped_root_app['duration_formatted'])):
            plt.text(value + 0.5, i, label, va='center', fontsize=10)
        plt.title('Время, проведенное в каждом приложении (root_app)')
        plt.xlabel('Время (минуты)')
        plt.ylabel('Приложение')
        plt.tight_layout()
        plt.show()

    if grouped_tab_title is not None and not grouped_tab_title.empty:
        grouped_tab_title = grouped_tab_title.sort_values('duration_minutes', ascending=False)
        grouped_tab_title['duration_formatted'] = grouped_tab_title['duration_minutes'].apply(lambda x: f"{int(x // 60)}:{int(x % 60):02d}")
        plt.figure(figsize=(14, 7))
        sns.barplot(x='duration_minutes', y='tab_title', data=grouped_tab_title, orient='h', palette="viridis", errorbar=None)
        for i, (value, label) in enumerate(zip(grouped_tab_title['duration_minutes'], grouped_tab_title['duration_formatted'])):
            plt.text(value + 0.1, i, label, va='center')
        plt.title('Время, проведенное в каждой вкладке (tab_title)')
        plt.xlabel('Время (минуты)')
        plt.ylabel('Вкладка')
        plt.tight_layout()
        plt.show()

    if context_duration_df is not None and not context_duration_df.empty:
        context_agg = pd.melt(context_duration_df, id_vars=['date'] + (['user_id'] if 'user_id' in context_duration_df.columns else []) + (['weekday'] if 'weekday' in context_duration_df.columns else []), value_vars=[col for col in context_duration_df.columns if col.startswith('context_')], var_name='context', value_name='duration_minutes')
        context_agg = context_agg.groupby('context')['duration_minutes'].sum().reset_index()
        context_agg = context_agg.sort_values('duration_minutes', ascending=False)
        context_agg['duration_formatted'] = context_agg['duration_minutes'].apply(
            lambda x: f"{int(x // 60)}:{int(x % 60):02d}"
        )

        plt.figure(figsize=(14, 7))
        sns.barplot(x='duration_minutes', y='context', data=context_agg,
                    orient='h', palette="viridis", errorbar=None)

        for i, (value, label) in enumerate(zip(context_agg['duration_minutes'], context_agg['duration_formatted'])):
            plt.text(value + 0.1, i, label, va='center')

        plt.title('Суммарное время в каждом контексте')
        plt.xlabel('Время (минуты)')
        plt.ylabel('Контекст')
        plt.tight_layout()
        plt.show()

    if grouped_tab_title_per_app is not None and not grouped_tab_title_per_app.empty:
        agg_data = grouped_tab_title_per_app.groupby(['root_app', 'tab_title'])['duration_minutes'].sum().reset_index()
        agg_data = agg_data.sort_values('duration_minutes', ascending=False)
        agg_data['duration_formatted'] = agg_data['duration_minutes'].apply(
             lambda x: f"{int(x // 60)}:{int(x % 60):02d}"
        )

        for app in agg_data['root_app'].unique():
            plt.figure(figsize=(14, 7))
            app_data = agg_data[agg_data['root_app'] == app]

            sns.barplot(x='duration_minutes', y='tab_title', data=app_data,
                    orient='h', palette="viridis", errorbar=None)

            for i, (value, label) in enumerate(zip(app_data['duration_minutes'], app_data['duration_formatted'])):
               plt.text(value + 0.1, i, label, va='center')

            plt.title(f'Время, проведенное в каждой вкладке внутри приложения {app}')
            plt.xlabel('Время (минуты)')
            plt.ylabel('Вкладка')
            plt.tight_layout()
            plt.show()

In [ ]:
def interactive_analysis(data):
    """Интерактивный выбор и визуализация."""
    data = preprocess_data(data)
    unique_dates = sorted(data['date'].unique().tolist())
    date_picker = widgets.DatePicker(description='Выберите дату:', options=unique_dates, disabled=False)
    unique_users = sorted(data['user_id'].unique().tolist())
    user_dropdown = widgets.Dropdown(options=['Все'] + unique_users, value='Все', description='Выберите пользователя:', disabled=False)
    # Убираем weekday_checkbox
    button = widgets.Button(description="Построить графики")

    # Используем Output виджет для управления выводом
    out = widgets.Output()

    def on_button_clicked(b):
        with out:  # Весь вывод идет в out
            clear_output(wait=True) # wait=True корректно работает с Output
            selected_date = date_picker.value
            selected_user = user_dropdown.value if user_dropdown.value != 'Все' else None
            # Убираем group_by_weekday
            filtered_data = data.copy()
            if selected_date:
                filtered_data = filtered_data[filtered_data['date'] == selected_date]
            if selected_user:
                filtered_data = filtered_data[filtered_data['user_id'] == selected_user]
            grouped_data = calculate_durations(filtered_data)  # Убираем group_by_weekday
            if grouped_data[0] is not None:
                visualize_durations_refined(*grouped_data)

    button.on_click(on_button_clicked)

    # Отображаем виджеты и *контейнер* для вывода
    display(widgets.VBox([date_picker, user_dropdown, button, out])) # Убираем weekday_checkbox

In [ ]:
# Пример использования (замените на ваш способ загрузки данных)
data = load_csv_from_drive('classified_user_data26.csv', base_path)  # Раскомментируйте и укажите ваш путь
data.head()

Первые 5 строк:
    id                               user_id  \
0  405  43db9537-0b46-47a1-937f-6d51af423cfe   
1  406  43db9537-0b46-47a1-937f-6d51af423cfe   
2  407  43db9537-0b46-47a1-937f-6d51af423cfe   
3  408  43db9537-0b46-47a1-937f-6d51af423cfe   
4  409  43db9537-0b46-47a1-937f-6d51af423cfe   

                          timestamp event_type                   record_id  \
0  2025-02-06 12:36:07.186000+03:00     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5   
1  2025-02-06 12:36:07.577000+03:00     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5   
2  2025-02-06 12:36:07.686000+03:00     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5   
3  2025-02-06 12:36:07.764000+03:00     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5   
4  2025-02-06 12:36:09.018000+03:00     VISUAL  01JKD9VP1Z8Z8H4S98797XEJD5   

  event_context                                        environment  \
0           NaN  {"mouse_x":0,"mouse_y":724,"modifiers":null,"l...   
1           NaN  {"mouse_x":265,"mouse_y":665,"modifiers":null,...   
2           NaN 

,id,user_id,timestamp,event_type,record_id,event_context,environment,root_app,tab_title,classname,...,modifiers,window_left,window_top,window_right,window_bottom,predicted_context,normalized_tab_title,category_scores,date,weekday
0,405,43db9537-0b46-47a1-937f-6d51af423cfe,2025-02-06 12:36:07.186000+03:00,VISUAL,01JKD9VP1Z8Z8H4S98797XEJD5,NaN,"{""mouse_x"":0,""mouse_y"":724,""modifiers"":null,""l...",['Google Chrome'],Курс ТЕХНОЛОГ Naumen Contact Center — Толк,Chrome_WidgetWin_1,...,NaN,-7,-7,1543,831,Обучение и саморазвитие,курс технолог naumen contact center толк,"{'Обучение и саморазвитие': 11, 'Финансы и бух...",2025-02-06,Thursday
1,406,43db9537-0b46-47a1-937f-6d51af423cfe,2025-02-06 12:36:07.577000+03:00,VISUAL,01JKD9VP1Z8Z8H4S98797XEJD5,NaN,"{""mouse_x"":265,""mouse_y"":665,""modifiers"":null,...",['Google Chrome'],Курс ТЕХНОЛОГ Naumen Contact Center — Толк,Chrome_WidgetWin_1,...,NaN,-7,-7,1543,831,Обучение и саморазвитие,курс технолог naumen contact center толк,"{'Обучение и саморазвитие': 11, 'Финансы и бух...",2025-02-06,Thursday
2,407,43db9537-0b46-47a1-937f-6d51af423cfe,2025-02-06 12:36:07.686000+03:00,VISUAL,01JKD9VP1Z8Z8H4S98797XEJD5,NaN,"{""mouse_x"":367,""mouse_y"":658,""modifiers"":null,...",['Google Chrome'],Курс ТЕХНОЛОГ Naumen Contact Center — Толк,Chrome_WidgetWin_1,...,NaN,-7,-7,1543,831,Обучение и саморазвитие,курс технолог naumen contact center толк,"{'Обучение и саморазвитие': 11, 'Финансы и бух...",2025-02-06,Thursday
3,408,43db9537-0b46-47a1-937f-6d51af423cfe,2025-02-06 12:36:07.764000+03:00,VISUAL,01JKD9VP1Z8Z8H4S98797XEJD5,NaN,"{""mouse_x"":368,""mouse_y"":658,""modifiers"":null,...",['Google Chrome'],Курс ТЕХНОЛОГ Naumen Contact Center — Толк,Chrome_WidgetWin_1,...,NaN,-7,-7,1543,831,Обучение и саморазвитие,курс технолог naumen contact center толк,"{'Обучение и саморазвитие': 11, 'Финансы и бух...",2025-02-06,Thursday
4,409,43db9537-0b46-47a1-937f-6d51af423cfe,2025-02-06 12:36:09.018000+03:00,VISUAL,01JKD9VP1Z8Z8H4S98797XEJD5,NaN,"{""mouse_x"":616,""mouse_y"":443,""modifiers"":null,...",['Google Chrome'],Курс ТЕХНОЛОГ Naumen Contact Center — Толк,Chrome_WidgetWin_1,...,NaN,-7,-7,1543,831,Обучение и саморазвитие,курс технолог naumen contact center толк,"{'Обучение и саморазвитие': 11, 'Финансы и бух...",2025-02-06,Thursday


In [ ]:
interactive_analysis(data)